# 💰 Profitability Analysis

This notebook implements the seventh dimension of our analytical framework: **Profitability Analysis**.

**Objective:** Shifting focus from top-line sales to net-line profitability (considering shipping, taxes, and discounts).

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('ggplot')

# Load data
df = pd.read_csv('../Amazon.csv')
success_orders = df[df['OrderStatus'].isin(['Delivered', 'Shipped'])].copy()

## 2. Defining 'Net Contribution'
Since we do not have Manufacturing Cost, we will look at 'Net Transactional Value' after Shipping and Taxes are removed from the Total Amount paid by the customer, which gives us the amount the merchant actually keeps for the goods sold.

In [ ]:
# Net Value = Customer Total - Shipping - Tax
success_orders['MerchantTake'] = success_orders['TotalAmount'] - success_orders['ShippingCost'] - success_orders['Tax']

total_paid = success_orders['TotalAmount'].sum()
total_shipping = success_orders['ShippingCost'].sum()
total_tax = success_orders['Tax'].sum()
total_merchant_take = success_orders['MerchantTake'].sum()

print(f"Total Paid by Customers: ${total_paid:,.2f}")
print(f"Total Shipping Costs: ${total_shipping:,.2f} ({(total_shipping/total_paid)*100:.2f}% of Total)")
print(f"Total Tax Collected: ${total_tax:,.2f} ({(total_tax/total_paid)*100:.2f}% of Total)")
print(f"Net Merchant Take-Home: ${total_merchant_take:,.2f} ({(total_merchant_take/total_paid)*100:.2f}% of Total)")

## 3. Shipping Cost Intensity by State
Identifying regions where shipping costs eat the most into revenue.

In [ ]:
state_shipping = success_orders.groupby('State').agg({
    'TotalAmount': 'sum',
    'ShippingCost': 'sum'
})
state_shipping['ShippingIntensity'] = (state_shipping['ShippingCost'] / state_shipping['TotalAmount']) * 100

plt.figure(figsize=(15, 6))
state_shipping['ShippingIntensity'].sort_values(ascending=False).plot(kind='bar', color='darkorange')
plt.title('Shipping Cost as % of Sales by State', fontweight='bold')
plt.ylabel('Shipping Intensity (%)')
plt.show()

## 4. Profitability Ranking - Categories

In [ ]:
cat_net = success_orders.groupby('Category')['MerchantTake'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 10))
plt.pie(cat_net, labels=cat_net.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('viridis', len(cat_net)))
plt.title('Distribution of Net Merchant Take-Home by Category', fontweight='bold')
plt.show()